## Harmony Py Library
### Job Steps Example

This notebook shows how to create a StepsRequest to interrogate intermediate results from a harmony service workflow. 

In [ ]:
import json
from harmony import BBox, Client, Collection, Request, Environment, StepsRequest

import helper
helper.install_project_and_dependencies('..')

In [ ]:
harmony_client = Client(env=Environment.UAT)  # assumes .netrc usage
request = Request(
    collection=Collection(id='C1268429309-EEDTEST'),
    granule_id=['G1281797307-EEDTEST'],
    format='image/tiff',
    variables=['Soil_Moisture_Retrieval_Data/landcover_class_fraction'],
    labels=['harmony-py-steps-example'],
)



In [ ]:
# submit an async request for processing and return the job_id
job_id = harmony_client.submit(request)
job_id

harmony_client.wait_for_processing(job_id, show_progress=True)


### Make a simple request to the steps endpoint

In [ ]:

step_request = StepsRequest(
    job_id=job_id, 
)
response = harmony_client.submit(step_request)
print(json.dumps(response, indent=2))

## Show the input and output files for the smap-l2-gridder step.


Find the work_item of the harmony-smap-l2-gridder step to filter the resolve files request.

In [ ]:
workitem_id = next(
    step["workItems"][0]["id"]
    for step in response["steps"]
    if "smap-l2-gridder" in step["serviceID"]
)
workitem_id

### Make the request to resolve the input and output files.

In [ ]:
step_files_request = StepsRequest(
    job_id=job_id, 
    work_item=[workitem_id],
    resolve_files=True, 
)
response = harmony_client.submit(step_files_request)
print(json.dumps(response, indent=2))

### Download intermediate files

`download_intermediate_files` resolves a work item's input and/or output files
and downloads them for you, returning a `Future` per file just like
`download_all`. To prevent accidentally downloading many files, it is limited
to the first 50 files, printing a warning if more than 50 files were resolved.


In [ ]:
futures = harmony_client.download_intermediate_files(
    job_id,
    work_items=[workitem_id],
)
downloaded_paths = [future.result() for future in futures]
downloaded_paths

#### Download more than 50 files 

If you must, you can bypass the 50 file limit on the `download_intermediate_files` helper, by downloading the files directly by first resolving them with a `StepsRequest` and then passing each URL to `client.download()`. 

In [ ]:
work_item = response["steps"][0]["workItems"][0]
all_urls = work_item.get("inputFiles", []) + work_item.get("outputFiles", [])

all_futures = [harmony_client.download(url) for url in all_urls]
all_paths = [future.result() for future in all_futures]
all_paths

## Paginating steps and resolved files

When a step has many `workItems`, or a `workItem` resolves to many input/output
files, the steps endpoint paginates. `StepsRequest` exposes the dynamically-named
page parameters as dictionaries keyed by the relevant index:

- `step_pages={step_index: page}` &rarr; `step<step_index>page=<page>`
- `work_item_input_pages={work_item_id: page}` &rarr; `workItem<work_item_id>inputPage=<page>`
- `work_item_output_pages={work_item_id: page}` &rarr; `workItem<work_item_id>outputPage=<page>`

Use `request_as_url` to see how the dictionaries translate into query parameters
without actually submitting the request, since it's invalid for our test case.

In [ ]:
paged_request = StepsRequest(
    job_id=job_id,
    work_item=[workitem_id],
    resolve_files=True,
    step_pages={2: 3},
    work_item_input_pages={workitem_id: 2},
    work_item_output_pages={workitem_id: 4},
)
print(harmony_client.request_as_url(paged_request))